In [43]:
import time
import warnings
from datetime import datetime
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

In [44]:
class MyTradingStrategy:
  def __init__(self, name):
    self.__name = name

  def generate_signal(self, price_data):
    print("This method is intended to be over ridden")
    return "Hold"

  @property
  def name(self):
    return self.__name

In [45]:
class MySMATradingStartegy(MyTradingStrategy):
  def __init__(self, swindow, lwindow): #For Short Moving average and Long Moving average
    self.__swindow = swindow #Short Window
    self.__lwindow = lwindow #Long Window
    super().__init__("My SMA Trading Strategy")

  def generate_signal(self, price_data):
    if len(price_data[-self.__lwindow:]) < self.__lwindow:
      return "Hold"
    short_avg = sum(price_data[-self.__swindow:]) / self.__swindow
    long_avg = sum(price_data[-self.__lwindow:]) / self.__lwindow

    if short_avg > long_avg:
      return "Buy"
    elif short_avg < long_avg:
      return "Sell"
    else:
      return "Hold"

  @property
  def swindow(self):
    return self.__swindow

  @property
  def lwindow(self):
    return self.__lwindow

In [46]:
class MyTrade:
  def __init__(self, strategy_name, signal, amount):
    self.__strategy_name = strategy_name
    self.__signal = signal
    self.__amount = amount
    self.__timestamp = datetime.now()

  def execute(self):
    print(f"Executed {self.__signal} trade with the strategy {self.__strategy_name} with amount {self.__amount} at the time {self.__timestamp}")


  @property
  def strategy_name(self):
    return self.__strategy_name

  @property
  def signal(self):
    return self.__signal

  @property
  def amount(self):
    return self.__amount

  @property
  def timestamp(self):
    return self.__timestamp


In [47]:
#Mock Traiding API
class MockTradingAPI:
    def __init__(self, balance):
        self.__balance = balance

    def place_order(self, trade, price):
        cost = price * trade.amount
        if trade.signal == "Buy" and self.__balance >= cost:
            self.__balance -= cost
            print(f"Placed a Buy trade at {price}, Remaining Balance: {self.__balance}")
        elif trade.signal == "Sell":
            self.__balance += cost
            print(f"Placed a Sell trade at {price}, Remaining Balance: {self.__balance}")
        else:
            print("Insufficient Balance or Invalid Signal")

    @property
    def balance(self):
        return self.__balance

In [ ]:
class MyTradingSystem:
  def __init__(self, api, strategy, symbol) -> None:
    self.__api = api
    self.__strategy = strategy
    self.__symbol = symbol
    self.__price_data = []

  def fetch_price_data(self):
    data = yf.download(tickers = self.__symbol, period='1d', interval='1m', auto_adjust=True, progress=False)
    if not data.empty:
      close = data['Close']
      if isinstance(close, pd.DataFrame):   # yfinance returns MultiIndex columns
          close = close.iloc[:, 0]
      price = float(close.iloc[-1])
      self.__price_data.append(price)
      if len(self.__price_data) > self.__strategy.lwindow:
        self.__price_data.pop(0)
        print(f"Fetched new Price Data: {price}")

  def run(self):
    self.fetch_price_data()
    signal = self.__strategy.generate_signal(self.__price_data)
    print(f"Generated Signal: {signal}")
    if signal in ("Sell", "Buy"):
      trade = MyTrade(self.__strategy.name, signal, 1)
      trade.execute()
      self.__api.place_order(trade, self.__price_data[-1])


  @property
  def api(self):
    return self.__api

  @property
  def strategy(self):
    return self.__strategy

  @property
  def symbol(self):
    return self.__symbol

  @property
  def balance(self):
    return self.__api.balance

In [59]:
symbol = "AAPL"
api = MockTradingAPI(balance=10000)
strategy = MySMATradingStartegy(3, 5)
system = MyTradingSystem(api, strategy, symbol)

for _ in range(12):
  system.run()
  print(f"Remaining balance: {api.balance}")
  time.sleep(2)

No data fetched
Generated Signal: Hold
Remaining balance: 10000
No data fetched
Generated Signal: Hold
Remaining balance: 10000
No data fetched
Generated Signal: Hold
Remaining balance: 10000
No data fetched
Generated Signal: Hold
Remaining balance: 10000
No data fetched
Generated Signal: Sell
Executed Sell trade with the strategy My SMA Trading Strategy with amount 1 at the time 2026-11-13 20:26:40.773793
Placed a Sell trade at 356.739990234375, Remaining Balance: 10356.739990234375
Remaining balance: 10356.739990234375
Fetched new Price Data: 356.7550048828125
Generated Signal: Sell
Executed Sell trade with the strategy My SMA Trading Strategy with amount 1 at the time 2026-11-13 20:26:42.895965
Placed a Sell trade at 356.7550048828125, Remaining Balance: 10713.494995117188
Remaining balance: 10713.494995117188
Fetched new Price Data: 356.7550048828125
Generated Signal: Sell
Executed Sell trade with the strategy My SMA Trading Strategy with amount 1 at the time 2026-11-13 20:26:45.00